In [1]:
import importlib
import processing_textblocks_helpers as pth
import os
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd
import json
import fitz
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import spacy_stanza
import re
import unicodedata
from spacy.tokens import Token, Doc
from spacy.language import Language
from spacy.symbols import ORTH
import spacy
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex


/home/jupyter-vojta/notebooks/labyrinth/venv_torch_nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-02 01:25:49 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-12-02 01:25:49 INFO: Loading these models for language: grc (Ancient_Greek):
| Processor | Package         |
-------------------------------
| tokenize  | proiel          |
| pos       | proiel_nocharlm |
| lemma     | proiel_nocharlm |
| depparse  | proiel_nocharlm |

2025-12-02 01:25:49 INFO: Using device: cuda
2025-12-02 01:25:49 INFO: Loading: tokenize
2025-12-02 01:25:50 INFO: Loading: pos
2025-12-02 01:25:50 INFO: Loading: lemma
2025-12-02 01:25:50 INFO: Loading: depparse
2025-12-0

In [2]:
importlib.reload(pth)

2025-12-02 01:25:56 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-12-02 01:25:56 INFO: Loading these models for language: grc (Ancient_Greek):
| Processor | Package         |
-------------------------------
| tokenize  | proiel          |
| pos       | proiel_nocharlm |
| lemma     | proiel_nocharlm |
| depparse  | proiel_nocharlm |

2025-12-02 01:25:56 INFO: Using device: cuda
2025-12-02 01:25:56 INFO: Loading: tokenize
2025-12-02 01:25:56 INFO: Loading: pos
2025-12-02 01:25:56 INFO: Loading: lemma
2025-12-02 01:25:56 INFO: Loading: depparse
2025-12-02 01:25:56 INFO: Done loading processors!


<module 'processing_textblocks_helpers' from '/home/jupyter-vojta/notebooks/EMLAP_ETL/scripts/processing_textblocks_helpers.py'>

In [3]:
source_path = "../data/emlap_sanitized_textblocks/"
len(os.listdir(source_path))

100

In [75]:
filenames = os.listdir(source_path)
filename = filenames[10]
print(filename)

100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json


In [76]:
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [77]:
textblocks[5][:10]

[{'coordinates': [526.0800170898438,
   172.10403442382812,
   1639.3897705078125,
   177.02403259277344],
  'text': 'EPISTOLA DEDICATORIA. ',
  'tag': 'header'},
 {'coordinates': [337.9200134277344,
   262.1040344238281,
   1993.8466796875,
   267.0240478515625],
  'text': 'Nullus quidem Paracelsita quicquam sibi commune cum schola ',
  'tag': 'text'},
 {'coordinates': [194.63999938964844,
   338.9038391113281,
   1991.999755859375,
   343.8238525390625],
  'text': 'Galenica esse prae se videtur ferre: aliqui tamen Hippocrati sunt aequio',
  'tag': 'text'},
 {'coordinates': [181.44000244140625,
   417.8399353027344,
   1995.948486328125,
   422.6399230957031],
  'text': 'res: Hic reijcit Chrysopoeian, ille magian, adiurationes, execrationes ec. ',
  'tag': 'text'},
 {'coordinates': [185.75999450683594,
   495.3838195800781,
   1972.9329833984375,
   500.3038330078125],
  'text': 'Puri puti Paracelsici omnia atque etiam sputa sui Monarchae exosculan',
  'tag': 'text'},
 {'coordinates':

In [78]:
doc, doc_margins = pth.process_with_source_tracking(textblocks[:], pth.nlp_latin)

In [79]:
work_id = filename[:6]
sent_dicts = pth.doc_to_sent_dicts(doc, work_id)
sent_dicts_margins = pth.doc_to_sent_dicts(doc_margins, work_id)

In [80]:
print(len(sent_dicts))
print(len(sent_dicts_margins))

25719
940


In [81]:
sent_dicts_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts, textblocks)
sent_dicts_margins_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts_margins, textblocks)

In [82]:
merged_sentences = pth.merge_main_and_margin_sentences(
    sent_dicts_with_coords,
    sent_dicts_margins_with_coords
)
len(merged_sentences)

26659

In [83]:
merged_sentences[:10]

[{'work_id': '100085',
  'sent_id': 0,
  'sent_text': 'COMMENTARIORUM',
  'tokens_data': [{'token_text': 'COMMENTARIORUM',
    'lemma': 'commentarius',
    'pos': 'X',
    'ref': {'page': [1], 'textblock': [1], 'tags': [], 'blocktype': 'text'},
    'char_start': 0,
    'char_end': 14,
    'coordinates': [74.63999938964844,
     332.4238586425781,
     1787.7001953125,
     337.3438720703125]}]},
 {'work_id': '100085',
  'sent_id': 1,
  'sent_text': 'ALCHYMIAE',
  'tokens_data': [{'token_text': 'ALCHYMIAE',
    'lemma': 'alchymiae',
    'pos': 'X',
    'ref': {'page': [1], 'textblock': [2], 'tags': [], 'blocktype': 'title'},
    'char_start': 0,
    'char_end': 9,
    'coordinates': [612.239990234375,
     450.7439270019531,
     1403.43505859375,
     455.6639404296875]}]},
 {'work_id': '100085',
  'sent_id': 2,
  'sent_text': 'ANDREAE LIBAUII MED.',
  'tokens_data': [{'token_text': 'ANDREAE',
    'lemma': 'Andreae',
    'pos': 'X',
    'ref': {'page': [1], 'textblock': [3], 'tags': []

In [84]:
merged_tokens = pd.DataFrame(
    (lambda d, n=n: (d.update({"sent_n": n}) or d))(t)
    for n, token_list in enumerate([sent["tokens_data"] for sent in merged_sentences])
    for t in token_list
)
for attr in ["page", "textblock", "tags", "blocktype"]:
    merged_tokens[attr] = merged_tokens.apply(lambda row: row["ref"][attr], axis=1)
merged_tokens.drop(["ref", "coordinates"], axis=1, inplace=True)
merged_tokens.head(5)

,token_text,lemma,pos,char_start,char_end,sent_n,page,textblock,tags,blocktype
0,COMMENTARIORUM,commentarius,X,0,14,0,[1],[1],[],text
1,ALCHYMIAE,alchymiae,X,0,9,1,[1],[2],[],title
2,ANDREAE,Andreae,X,0,7,2,[1],[3],[],text
3,LIBAUII,Libauii,NOUN,8,15,2,[1],[3],[],text
4,MED,ego,ADJ,16,19,2,[1],[3],[],text


In [85]:
len(merged_tokens)

394711

In [86]:
[type(x) for x in merged_tokens["tags"].tolist()]

[list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,
 list,

In [87]:
merged_tokens[merged_tokens["tags"].apply(lambda x: len(x) > 0)]["tags"].apply(lambda x: x[0]).value_counts()

tags
GR    2828
G      342
S      297
I        2
Name: count, dtype: int64

In [112]:
def process_textblocks(textblocks):
    filepath = os.path.join(source_path, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        textblocks = json.load(f)
    doc, doc_margins = pth.process_with_source_tracking(textblocks[:], pth.nlp_latin)
    work_id = filename[:6]
    sent_dicts = pth.doc_to_sent_dicts(doc, work_id)
    sent_dicts_margins = pth.doc_to_sent_dicts(doc_margins, work_id)
    sent_dicts_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts, textblocks)
    sent_dicts_margins_with_coords = pth.attach_coordinates_to_sent_dicts(sent_dicts_margins, textblocks)
    merged_sentences = pth.merge_main_and_margin_sentences(
        sent_dicts_with_coords,
        sent_dicts_margins_with_coords
    )
    merged_sentences_uncoord = []
    for sent_data in merged_sentences:
        for token_data in sent_data["tokens_data"]:
            del token_data["coordinates"]
    return merged_sentences

In [94]:
sents_data = process_textblocks(textblocks)

In [109]:
target_path = "../data/sents_data_jsons_dicts/"
os.makedirs(target_path, exist_ok=True)

In [113]:
for filename in os.listdir(source_path):
    id = filename[:6]
    outfile = id + ".json"
    if outfile not in os.listdir(target_path):
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        sent_dicts = process_textblocks(textblocks)
        outpath = os.path.join(target_path, outfile)
        with open(outpath, 'w', encoding='utf-8') as f:
            json.dump(sent_dicts, f, ensure_ascii=False, indent=2)